# Editorial interventions to TEI with an LLM on the DHinfra cluster

In the edition of the Hammer-Purgstall correspondence, editors marked their interventions in
the transcriptions with square brackets: `Fr[au]` (an abbreviation expanded), `[ein]` (a word
supplied), `gefühltes[?]` (an uncertain reading), `[Lücke]` (a gap). The brackets say that
something was done, not what. The task here: for each bracketed sequence, name the type of
intervention and give its TEI encoding.

Ten text snippets (German, English, French, Italian) go in, one JSON answer per snippet comes
out: `{"d[en]": {"type": "Abbreviation", "tei": "<choice><abbr>d</abbr><expan>den</expan></choice>"}}`.
(The line above only shows the format. It is not a result.)

This notebook uses the same helper module and the same gateway as `01_letters_to_tei.ipynb`.
Here the input is not a text file but a JSON file, and the model answers in JSON.

## 1. Set up

In [ ]:
# Run once per session. The cluster's Python environment may already have these.
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

In [ ]:
import json
from pathlib import Path
import concurrent.futures as cf

import pandas as pd

import helpers

MODEL = "deepseek-v4.1-flash"   # pick a name from the list printed below
THINKING = False                # True: the model reasons first (slower, more tokens)
WORKERS = 4                     # parallel requests; keep this small, the cluster is shared

PROMPT_DIR = Path("prompts/editorial_interventions")
INPUT_FILE = Path("data/editorial_interventions/sample.json")
OUTPUT_DIR = Path("output") / MODEL
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

client = helpers.get_client()
print([m.id for m in client.models.list().data])

## 2. The data and the prompt

Each snippet has an `element_id` (which element of which letter it comes from), the text of
that element as `context`, and the `bracketed_sequences` in it. A snippet can contain
several sequences; they are sent together, because the surrounding text is the same.

In [ ]:
snippets = json.loads(INPUT_FILE.read_text(encoding="utf-8"))
pd.DataFrame(snippets)[["element_id", "bracketed_sequences", "context"]]

The system prompt has three parts: `prompt.txt` (the task), `encoding_rules.txt` (six
intervention types and the TEI element for each) and `few_shot_examples.txt` (five examples with
the expected JSON). The user message is one instruction plus the context and the sequences.

In [ ]:
system_prompt = helpers.build_system_prompt(PROMPT_DIR)
print(f"{len(system_prompt):,} characters")
print(system_prompt[:1800], "...")

## 3. One snippet

Snippet 2 has two sequences (`d[en]` and `Septemb[er] [17]98`) in a dateline.

In [ ]:
snippet = snippets[1]
message = helpers.build_items_message(PROMPT_DIR, snippet["context"], snippet["bracketed_sequences"])
print(message)

In [ ]:
raw, usage = helpers.ask_model(client, MODEL, system_prompt, message, thinking=THINKING)
print(raw)
print()
print("tokens in/out:", usage.prompt_tokens, usage.completion_tokens)

## 4. All ten snippets

One request per snippet, a few at a time. The raw answers are saved to
`output/<model>/editorial_interventions.json`.

In [ ]:
def run_one(snippet):
    message = helpers.build_items_message(
        PROMPT_DIR, snippet["context"], snippet["bracketed_sequences"])
    raw, usage = helpers.ask_model(client, MODEL, system_prompt, message, thinking=THINKING)
    return snippet["element_id"], raw, usage.prompt_tokens, usage.completion_tokens

answers = {}
with cf.ThreadPoolExecutor(max_workers=WORKERS) as pool:
    futures = {pool.submit(run_one, s): s for s in snippets}
    for future in cf.as_completed(futures):
        element_id = futures[future]["element_id"]
        try:
            answers[element_id] = future.result()
            print("done  ", element_id)
        except Exception as err:   # keep going; one failed snippet should not stop the batch
            print("FAILED", element_id, "-", err)

saved = {eid: helpers.parse_json_answer(raw) for eid, (_, raw, _, _) in answers.items()}
(OUTPUT_DIR / "editorial_interventions.json").write_text(
    json.dumps(saved, ensure_ascii=False, indent=2), encoding="utf-8")

## 5. Check the results

There is no reference encoding in this example. The checks are cheap ones, one row per
bracketed sequence:

- **answered**: the answer contains the sequence.
- **type_in_list**: the type the model named is one of the six from the encoding rules (two types joined by a semicolon are allowed). This does *not* say that it is the right type.
- **tei_ok**: the TEI is well-formed, uses only the elements from the rules and has no attributes. This does *not* say that the encoding is right.

Whether the *type* and the *encoding* are right needs your own reading of the table.

In [ ]:
rows = []
for snippet in snippets:
    eid = snippet["element_id"]
    answer = saved.get(eid)
    for row in helpers.check_interventions(answer, snippet["bracketed_sequences"]):
        rows.append({"element_id": eid, **row})
table = pd.DataFrame(rows)
pd.set_option("display.max_colwidth", 80)
table

In [ ]:
print(f"{int(table.answered.sum())}/{len(table)} sequences answered, "
      f"{int(table.type_in_list.sum())} with a type from the list, {int(table.tei_ok.sum())} with valid TEI")

## Where next

- Change `MODEL`, or the rules in `prompts/editorial_interventions/`, and run again. The output goes
  to a folder named after the model.
- Use your own snippets: a JSON list with `element_id`, `context` and `bracketed_sequences`.
- To put an encoding back into the full TEI document, the `element_id` and the sequence tell you where it belongs.